In [1]:
import os
from pyspark.sql import SparkSession

# The Spark Connect server has no catalogs of its own: each session registers the Polaris catalogs
# it uses, authenticated with the user's OIDC token, which JupyterHub injects into the pod.
OIDC_TOKEN = os.environ["OIDC_TOKEN"]
POLARIS_DOMAIN_NAME = os.environ.get("POLARIS_DOMAIN_NAME", "emryspolaris.click")
OIDC_DOMAIN_NAME = os.environ.get("OIDC_DOMAIN_NAME", "emrysoauth.click")


def with_catalog(builder, name, warehouse=None):
    """Registers the Polaris catalog warehouse (default: name) as the Spark catalog name."""
    prefix = f"spark.sql.catalog.{name}"
    return (builder
            .config(prefix, "org.apache.iceberg.spark.SparkCatalog")
            .config(f"{prefix}.type", "rest")
            .config(f"{prefix}.uri", f"https://{POLARIS_DOMAIN_NAME}/api/catalog")
            .config(f"{prefix}.warehouse", warehouse or name)
            .config(f"{prefix}.cache-enabled", "false")
            .config(f"{prefix}.header.X-Iceberg-Access-Delegation", "vended-credentials")
            .config(f"{prefix}.oauth2-server-uri", f"https://{OIDC_DOMAIN_NAME}/realms/polaris/protocol/openid-connect/token")
            .config(f"{prefix}.scope", "openid")
            .config(f"{prefix}.token", OIDC_TOKEN))


builder = SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002")
for catalog in ["aws", "azure", "gcp"]:
    builder = with_catalog(builder, catalog)
spark = builder.getOrCreate()

print("session created")

df_test = spark.createDataFrame([["test", 1]], ['name', 'num'])
print("Initialized test data")


session created
Initialized test data


![](https://media.licdn.com/dms/image/v2/D4E0BAQGumdWW8R2giQ/company-logo_200_200/B4EZ87y9pLK4AE-/0/1783414643504/health_data_research_service_logo?e=2147483647&v=beta&t=GxsLjCFo9AmdCTHkDLK622NOIF3hplTcipdZBeJbzPY)

In [2]:
from datetime import datetime
import pandas as pd

In [3]:
aws_cond_df = spark.createDataFrame(pd.read_csv('conditions.csv'))
aws_obs_df = spark.createDataFrame(pd.read_csv('observations.csv'))
aws_pats_df = spark.createDataFrame(pd.read_csv('patients.csv'))





In [6]:



#
# AWS
#
# discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
aws_namespace = "aws.cancer"
aws_tables = [(aws_cond_df,"conditions"),
              (aws_obs_df,"observations"),
              (aws_pats_df,"patients")]

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {aws_namespace}")
print(f"Created namespace {aws_namespace}")

for df,t in aws_tables:
    aws_fqn = f"{aws_namespace}.{t}" 
    df.writeTo(aws_fqn).create()
    print(f"Wrote {aws_fqn} to AWS")

# spark.sql(f"SELECT * FROM {aws_fqn}").show()


UnknownException: (org.apache.iceberg.exceptions.RESTException) Error occurred while processing POST request

JVM stacktrace:
org.apache.iceberg.exceptions.RESTException
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:359)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:297)
	at org.apache.iceberg.rest.BaseHTTPClient.postForm(BaseHTTPClient.java:136)
	at org.apache.iceberg.rest.auth.OAuth2Util.fetchToken(OAuth2Util.java:291)
	at org.apache.iceberg.rest.auth.OAuth2Manager.initSession(OAuth2Manager.java:87)
	at org.apache.iceberg.rest.auth.OAuth2Manager.initSession(OAuth2Manager.java:40)
	at org.apache.iceberg.rest.RESTSessionCatalog.initialize(RESTSessionCatalog.java:204)
	at org.apache.iceberg.rest.RESTCatalog.initialize(RESTCatalog.java:82)
	at org.apache.iceberg.CatalogUtil.loadCatalog(CatalogUtil.java:280)
	at org.apache.iceberg.CatalogUtil.buildIcebergCatalog(CatalogUtil.java:337)
	at org.apache.iceberg.spark.SparkCatalog.buildIcebergCatalog(SparkCatal

In [ ]:
spark.sql("describe table extended aws.acceptancetests.ouh_conditions").show(truncate=False)
spark.sql("describe table extended aws.acceptancetests.ouh_conditions").filter("col_name = 'Location'").show(truncate=False)
spark.sql("show tblproperties aws.acceptancetests.ouh_conditions").show(truncate=False)
